In [1]:
import climt
from sympl import DataArray, TendencyComponent, AdamsBashforth
from sympl import (
    PlotFunctionMonitor, NetCDFMonitor,
    TimeDifferencingWrapper, UpdateFrequencyWrapper,
    set_constant, get_constant, initialize_numpy_arrays_with_properties
)
import gfs_dynamical_core
from datetime import timedelta
import numpy as np
import torch
import torch.nn as nn
import sys
sys.path.append("..")
from models import DynamicMLP

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/projects/sds-lab/Shuochen/miniconda3/envs/ai/lib/python3.9/site-packages/climt/_core/initialization.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [8]:
class TorchScaler:
    def __init__(self, mean, std):
        self.mean = mean  # expected shape: (C, 1, 1)
        self.std = std
    
    def transform(self, x):
        # make sure mean/std have shape (C,1,1) for broadcasting
        mean = self.mean
        std = self.std
        if isinstance(mean, torch.Tensor):
            mean = mean.cpu().numpy()
        if isinstance(std, torch.Tensor):
            std = std.cpu().numpy()
        mean = mean.reshape(-1, 1, 1)  # broadcast along H, W
        std = std.reshape(-1, 1, 1)
        return (x - mean) / std
    
    def inverse_transform(self, x):
        mean = self.mean
        std = self.std
        if isinstance(mean, torch.Tensor):
            mean = mean.cpu().numpy()
        if isinstance(std, torch.Tensor):
            std = std.cpu().numpy()
        mean = mean.reshape(-1, 1, 1)
        std = std.reshape(-1, 1, 1)
        return x * std + mean

class NNParameterization(TendencyComponent):

    input_properties = {
        'air_temperature': {'dims': ['mid_levels', 'lat', 'lon'], 'units': 'degK'},
        'specific_humidity': {'dims': ['mid_levels', 'lat', 'lon'], 'units': 'kg/kg'},
        'surface_air_pressure': {'dims': ['lat', 'lon'], 'units': 'Pa'},
        'surface_upward_latent_heat_flux': {'dims': ['lat', 'lon'], 'units': 'W m^-2'},
        'surface_upward_sensible_heat_flux': {'dims': ['lat', 'lon'], 'units': 'W m^-2'},
    }

    tendency_properties = {
        'air_temperature_tendency_from_longwave': {
            'dims': ['mid_levels', 'lat', 'lon'],
            'units': 'degK day^-1'
        }
    }
    
    diagnostic_properties = {}

    def __init__(self):
        super().__init__()

        self.IN_FEATURES = 59
        self.OUT_FEATURES = 28

        ckpt = torch.load("./best_model/best_model_trial_259.pth",map_location=device)
        ckpt_norm = torch.load('/projects/sds-lab/Shuochen/climt/gmd_aquaplanet/64x32_4h_normalization.pth', map_location=device)
        self.input_scaler  = TorchScaler(ckpt_norm['X_mean'], ckpt_norm['X_std'])
        self.output_scaler = TorchScaler(ckpt_norm['y_mean'], ckpt_norm['y_std'])

        self.model = DynamicMLP(
            self.IN_FEATURES,
            self.OUT_FEATURES,
            ckpt["hidden_sizes"]
        ).to(device)

        self.model.load_state_dict(ckpt["model_state"])
        self.model.eval()

    def array_call(self, state):

        tendencies = initialize_numpy_arrays_with_properties(
            self.tendency_properties, state, self.input_properties
        )
        diagnostics = initialize_numpy_arrays_with_properties(
            self.diagnostic_properties, state, self.input_properties
        )
        # ---------------------------------
        # Extract state
        # ---------------------------------
        T = state['air_temperature']        # (L, H, W)
        q = state['specific_humidity']      # (L, H, W)
        ps = state['surface_air_pressure']  # (H, W)
        lh = state['surface_upward_latent_heat_flux'] # (H, W)
        sh = state['surface_upward_sensible_heat_flux'] # (H, W)
        # ---------------------------
        # Build NN input [C, H, W]
        # ---------------------------
        L, H, W = T.shape
        ncol = H * W
        
        x = np.concatenate([
            T.reshape(-1, *ps.shape),
            q.reshape(-1, *ps.shape),
            ps[None, ...],
            lh[None, ...],
            sh[None, ...],
        ], axis=0)  # (IN_FEATURES, H, W)
        
        # normalize
        if self.input_scaler is not None:
            x = self.input_scaler.transform(x)
            
        # Add batch dimension
        x = torch.tensor(x, dtype=torch.float32, device=device).unsqueeze(0)
        # ---------------------------------
        # NN inference
        # ---------------------------------
        with torch.no_grad():
            y = self.model(x)   # (ncol, 28)
        y = y.cpu().numpy().squeeze()
        # inverse normalize if needed
        if self.output_scaler is not None:
            y = self.output_scaler.inverse_transform(y)
            
        # ---------------------------------
        # Map NN output → temperature tendency
        # ---------------------------------
        
        # print(y.shape)
        # tendencies['air_temperature_tendency_from_longwave'][:] = y[:L, :, :]
        # return tendencies, diagnostics
        # Convert from K/day → K/s
        # dT = y / 86400.0  # 86400 seconds in a day
        
        # da = DataArray(dT, dims=['mid_levels', 'lat', 'lon'])
        # # da.attrs['units'] = 'degK s^-1'
        # tendencies = {
        #     'air_temperature_tendency_from_longwave': da
        # }
        # tendencies['air_temperature_tendency_from_longwave'][:] = y
        # da = DataArray(y, dims=['mid_levels', 'lat', 'lon'])
        # # da.attrs['units'] = 'degK day^-1'
        # tendencies = {'air_temperature_tendency_from_longwave': da}
        # tendencies['air_temperature_tendency_from_longwave'][:] = y[:L]
        # tendencies['air_temperature_tendency_from_longwave'].attrs['units'] = 'degK'
        da = DataArray(y, dims=['mid_levels', 'lat', 'lon'])
        da.attrs['units'] = 'degK day^-1'
        tendencies = {'air_temperature_tendency_from_longwave': da}
        
        return tendencies, diagnostics
        
model_time_step = timedelta(minutes=10)
# Create components
convection = climt.EmanuelConvection(tendencies_in_diagnostics=True)
simple_physics = TimeDifferencingWrapper(climt.SimplePhysics())
radiation_step = timedelta(hours=1)
radiation_lw = UpdateFrequencyWrapper(
    climt.RRTMGLongwave(), radiation_step)
radiation_sw = UpdateFrequencyWrapper(
    climt.RRTMGShortwave(), radiation_step)
slab_surface = climt.SlabSurface()
# nn component
nn_component = NNParameterization()

dycore = gfs_dynamical_core.GFSDynamicalCore(
    [simple_physics, slab_surface, radiation_sw,
     radiation_lw, convection], number_of_damped_levels=5
)
# dycore = gfs_dynamical_core.GFSDynamicalCore(
#     [simple_physics, slab_surface, radiation_sw,
#      nn_component, convection], number_of_damped_levels=5
# )

grid = climt.get_grid(nx=64, ny=32)
my_state = climt.get_default_state([dycore], grid_state=grid)


timestep = timedelta(minutes=10)
time_stepper = AdamsBashforth([nn_component])
n_steps = 144  # 1 day at 10-min timestep
for i in range(n_steps):

    diag, my_state = dycore(my_state, model_time_step)
    my_state.update(diag)
    # print(my_state['air_temperature'].attrs)
    # ?
    diag['air_temperature_tendency_from_longwave'].attrs['units'] = 'degK'
    # diagnostics, my_state = time_stepper(my_state, model_time_step)
    # print(my_state.keys())
    
    if i % 10 == 0:
        T_mid = my_state['air_temperature'].values[15].mean()
        print(f"Step {i}, mean T_mid = {T_mid:.2f} K")

InvalidPropertyDictError: Incompatibility between dims of quantity air_temperature: dims {'mid_levels', 'lon', 'lat'} and {'mid_levels'} are incompatible